# Motor Cortex Dynamics — reproduction + interactive viewer
# 운동피질 동역학 — 재현 + 인터랙티브 뷰어

**Stage 1 (descriptive):** PCA + hand-implemented **jPCA** on real M1 data → rotational dynamics (Churchland et al. 2012).

**Stage 2 (mechanistic):** **fixed-point analysis** of a task-trained RNN → the local structure that *generates* the rotation (Sussillo & Barak 2013; Sussillo et al. 2015).

jPCA is *descriptive* — one global skew-symmetric linear fit to trajectory geometry, needs no equations, runs on **brain and RNN**. Fixed-point analysis is *mechanistic* — needs the evaluable vector field `dx/dt`, so it runs on the **RNN only**. Fixed points **generate** the rotation jPCA **describes**. The viewer keeps this asymmetry visible: brain = trajectories + jPCA plane; RNN = trajectories + jPCA + fixed points + flow field.

Each section follows: **(1)** markdown math + *why* → **(2)** code → **(3)** inline plotly → **(4)** markdown interpretation + failure modes.

---

**한국어 설명.** 1단계(기술적): 실제 M1 데이터에 PCA + 직접 구현한 jPCA → 회전 동역학. 2단계(기전적): 과제로 학습한 RNN의 고정점 분석 → 그 회전을 *생성하는* 국소 구조. jPCA는 방정식이 필요 없어 뇌·RNN 모두에, 고정점 분석은 벡터장 `dx/dt`가 필요해 RNN에만 적용된다. 뷰어는 이 비대칭을 드러낸다: 뇌 = 궤적 + jPCA 평면 / RNN = 궤적 + jPCA + 고정점 + 흐름장.

## 학습 로드맵 / How to study this notebook

Read **top to bottom** — the notebook is ordered so implementation and learning
unfold together. Every milestone repeats the same five-step rhythm:

1. **수식/이유 (math + why)** — what we compute and *why this choice, what we rejected*.
2. **🔍 구현 핵심 (under the hood)** — the *actual* source of the crux function,
   printed inline with `inspect.getsource`. (Canonical logic lives in `python/`
   per the project rule; it is shown here so you can read the implementation in
   order without leaving the notebook.)
3. **실행 (run)** — call it and print the falsifiable numbers.
4. **그림 (plotly)** — see the result.
5. **해석 (interpretation)** — what it means + the failure modes we watched.

**The one idea that governs everything** (internalize this first): jPCA is
*descriptive* — it needs only trajectories, so it runs on **brain and RNN**.
Fixed-point analysis is *mechanistic* — it needs the equations `dx/dt`, so it runs
on the **RNN only**. Fixed points **generate** the rotation jPCA **describes**.

**Suggested order:** skim this roadmap → run "0 · Environment setup" → then
M0 → M1 → M2 → M3 → M4 → M5 in sequence. M0 (jPCA) and M1 (the fixed-point finder
on a *known-answer* task) are the two foundations; everything after builds on them.

---

**한국어.** **위에서 아래로** 읽으세요 — 구현과 학습이 순서대로 이어지도록 배치했습니다.
각 마일스톤은 같은 5단계 리듬을 반복합니다: **① 수식/이유 → ② 🔍 구현 핵심(핵심 함수의
실제 소스를 `inspect`로 인라인 표시; 정식 로직은 `python/`에 있고 학습용으로 여기 보여줌)
→ ③ 실행(숫자 출력) → ④ 그림 → ⑤ 해석(+관찰한 실패 모드).**

**가장 먼저 체화할 한 가지:** jPCA는 *기술적* — 궤적만 필요해 **뇌·RNN 둘 다**. 고정점
분석은 *기전적* — 방정식 `dx/dt`가 필요해 **RNN에만**. 고정점이 jPCA가 *기술*하는 회전을
*생성*합니다.

**권장 순서:** 이 로드맵 훑기 → "0 · 환경 설정" 실행 → M0 → M1 → M2 → M3 → M4 → M5.
M0(jPCA)와 M1(정답 아는 과제의 고정점 탐색기)이 두 토대이고, 이후는 그 위에 쌓입니다.

## 0 · Environment setup / 환경 설정
Mount Drive first (Colab filesystem is ephemeral), then install deps and import the `python/` modules.
**한국어:** Colab 파일시스템은 휘발성이라 먼저 Drive를 마운트하고, 의존성 설치 후 `python/` 모듈을 임포트한다.

In [ ]:
# --- FIRST CELL: put the repo on the path so `python/`, `tests/`, `scripts/` import ---
# 첫 셀: 리포를 경로에 올려 python/·tests/·scripts/ 를 임포트 가능하게 만든다.
# WHY: Colab has ONLY this notebook, not the repo — so we clone it. Without this you get
# "ModuleNotFoundError: No module named 'python'". Drive is mounted so trained weights /
# downloads persist across sessions (Colab disk is ephemeral).
from pathlib import Path
import os, sys, subprocess

REPO_URL = 'https://github.com/duck-bin/neural-dynamics.git'
BRANCH   = 'claude/jpca-fixed-point-rnn-kzwz05'   # after merge, use 'main'

try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = Path('/content/neural-dynamics')
    if not (REPO / 'python').exists():          # clone once per session (private repo: authenticate)
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(REPO)], check=True)
    # persist cache on Drive: symlink REPO/cache -> Drive so scripts (which write REPO/cache) survive restarts
    drive_cache = Path('/content/drive/MyDrive/neural-dynamics/cache'); drive_cache.mkdir(parents=True, exist_ok=True)
    if not (REPO / 'cache').exists():
        os.symlink(drive_cache, REPO / 'cache')
else:
    REPO = Path.cwd().resolve()                  # local: walk up to the repo root from anywhere
    while REPO != REPO.parent and not (REPO / 'python').exists():
        REPO = REPO.parent

os.chdir(REPO)                                    # so relative paths (cache/, data/) resolve
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))                 # so `from python import ...` / `from tests import ...` work
CACHE_DIR = REPO / 'cache'; DATA_DIR = REPO / 'data'
CACHE_DIR.mkdir(parents=True, exist_ok=True); DATA_DIR.mkdir(parents=True, exist_ok=True)
assert (REPO / 'python').exists(), f'repo not found at {REPO} — check the clone step'
print(f'IN_COLAB={IN_COLAB}  REPO={REPO}  (python/ importable: {(REPO/"python").exists()})')

In [ ]:
# --- Dependencies — auto-installs in Colab; a local venv already has them. Run once per session. ---
# 의존성 — Colab에선 자동 설치, 로컬 venv엔 이미 있음. 세션당 한 번.
if IN_COLAB:
    # numpy/scipy/scikit-learn/torch/plotly ship with Colab; ensure + add the differential-test oracles.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'scipy', 'scikit-learn', 'plotly'], check=False)
    # jPCA reference oracle (M0/M2 differential test):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                    'git+https://github.com/bantin/jPCA.git'], check=False)
    # fixed-point reference (M1 differential test): clone + expose via REF_FPA_DIR
    ref = REPO.parent / 'pytorch-fixed-point-analysis'
    if not ref.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/tripdancer0916/pytorch-fixed-point-analysis.git', str(ref)], check=False)
    os.environ['REF_FPA_DIR'] = str(ref)
print('deps ready (IN_COLAB=%s)' % IN_COLAB)

In [ ]:
# --- Sanity check: the pipeline modules import (logic lives in python/, not in cells) ---
# 임포트 확인: 파이프라인 모듈이 잘 불러와지는지 (로직은 셀이 아니라 python/ 에 있다).
from python import data, pca_jpca, rnn, fixedpoints, flowfield, export  # noqa: F401
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'notebook'
print('modules imported OK from', REPO)

## M0 · jPCA hand implementation _(PASS — tests/test_m0_jpca.py)_
**(1) math/why:** soft-normalize → subtract cross-condition mean per timepoint (Lebedev 2019 critique noted, kept for faithful reproduction) → PCA (k=6) → finite-diff `Ẋ` → fit `Ẋ = X Mᵀ` with `M = -Mᵀ` (skew-symmetric constrained least squares, **closed form** via a vectorized skew basis) → eigendecompose (±iω) → top plane.
**PASS/FAIL:** fit R² of `Ẋ=X Mᵀ`; top rotation-plane variance fraction; principal angle vs Antin `jPCA` < 5°, freq within 5%. The hand solver is closed-form; Antin's is iterative CG — two different methods for one objective, so agreement is independent corroboration.

**한국어.** 소프트 정규화 → 조건간 평균 차감 → PCA(k=6) → 유한차분 → 반대칭 제약 최소제곱(닫힌 형태) → 고유분해 → 최상위 회전 평면. 합격 기준: 적합 R², 회전 평면 분산 비율, Antin 대비 주각 < 5°·주파수 < 5%.

In [ ]:
# 🔍 구현 핵심 (under the hood) — jPCA의 크럭스: 반대칭 제약 최소제곱 + 고유분해.
# The crux of jPCA: the skew-symmetric constrained solve (step 5) and the eigendecomposition (step 6).
# Canonical code lives in python/pca_jpca.py — shown here so you read it in order.
import inspect
from python import pca_jpca as J
print(inspect.getsource(J.fit_skew_symmetric_M))   # step 5: vectorized closed-form skew LS
print(inspect.getsource(J.jpc_plane_from_M))        # step 6: top +/- i*omega plane

In [ ]:
# (2) Run the hand pipeline + differential test vs Antin (known-answer synthetic here; M2 re-runs on real data).
from python import pca_jpca as J
from tests.test_m0_jpca import make_synthetic
datas, omega_true = make_synthetic()          # (C, T, N) condition-averaged rates
res = J.jpca(datas, num_pcs=6, subtract_ccm=True)
print(f"fit R2 (Xdot = X M^T, skew)   : {res.fit_R2:.4f}")
print(f"top rotation-plane variance   : {res.plane_var_frac:.4f}")
print(f"recovered omega [rad/bin]     : {res.omega:.4f}  (injected {omega_true}; forward-diff recovers sin(w))")
# Full hand-vs-Antin differential test:  %run tests/test_m0_jpca.py

In [ ]:
# (3) Inline plotly: trajectories on the top jPC plane should sweep out rotations.
import plotly.graph_objects as go
fig = go.Figure()
for c in range(res.projected.shape[0]):
    xy = res.projected[c]
    fig.add_trace(go.Scatter(x=xy[:, 0], y=xy[:, 1], mode='lines', line=dict(width=2), name=f'cond {c}'))
    fig.add_trace(go.Scatter(x=[xy[0, 0]], y=[xy[0, 1]], mode='markers',
                             marker=dict(size=8, line=dict(width=0.5, color='black')), showlegend=False))
fig.update_layout(template='plotly_dark', width=560, height=520,
                  title=f'Top jPC plane — rotations (R2={res.fit_R2:.2f})', xaxis_title='jPC1', yaxis_title='jPC2')
fig.update_yaxes(scaleanchor='x', scaleratio=1)   # equal aspect
fig.show()

**(4) Interpretation / 해석.** Hand solver matches Antin's `M` to ~4e-5 and the jPC planes agree to **<0.001°** — two independent solvers (our closed-form vectorized LS vs Antin's iterative CG) of the same skew-symmetric objective. Fit R² (~0.86 synthetic) sits below the unconstrained ceiling (~0.99): that gap is what the skew constraint *costs*, because real data also carries non-rotational (expansion/decay) structure a pure rotation cannot fit — measuring that gap is the point of jPCA. Recovered frequency is `sin(ω)` by construction of the forward difference (~1.5% bias at 0.3 rad/bin, shared by the oracle; kept for faithful reproduction).

**한국어.** 직접 구현 해는 Antin의 반복 CG 해와 `M` 기준 ~4e-5, 평면 <0.001°로 일치. R²(~0.86)가 무제약 상한(~0.99)보다 낮은 것은 반대칭 제약의 *비용*(회전이 담을 수 없는 팽창/감쇠). 주파수는 전진차분 특성상 `sin(ω)` — 알려진 편향, 레퍼런스도 동일.

## M1 · Flip-flop calibration (known answer — DO FIRST) _(PASS — tests/test_m1_flipflop.py)_
**(1) math/why:** train a small tanh RNN on the 3-bit flip-flop; run the fixed-point finder on the autonomous map residual `F(h) = tanh(W_in·0 + W_hh h + b) − h`, minimizing `q = ½‖F‖²`.
**FALSIFICATION GATE:** exactly **8 stable fixed points** near cube corners, else STOP and debug before touching reaching data. Differential test vs `pytorch-fixed-point-analysis`; render 8 FPs + trajectories.

**한국어.** 3-bit 플립플롭으로 tanh RNN 학습 → 맵 잔차 `F(h)`에 `q=½‖F‖²` 최소화. 반증 게이트: 큐브 꼭짓점의 안정 고정점 정확히 8개. 아니면 STOP.

In [ ]:
# 🔍 구현 핵심 — 자율계 속도 F(h)와 고정점 탐색기(q=½||F||² 최소화).
# The autonomous velocity F(h) = tanh(W_in u + W_hh h + b) - h, and the finder that
# minimizes q = ½||F||² from ICs sampled on visited states (Adam + optional L-BFGS).
import inspect
from python import fixedpoints as fp
print(inspect.getsource(fp.discrete_velocity))
print(inspect.getsource(fp.find_fixed_points))

In [ ]:
# (2) Train the flip-flop RNN, then find its fixed points.
import numpy as np, torch
from python import rnn, fixedpoints as fp
model, mse, _ = rnn.train_flipflop(n_hid=64, iters=1200, batch=64, length=100, lr=3e-3, seed=0)
print(f"flip-flop task MSE: {mse:.5f}  (solved if < 0.02)")
for p in model.parameters():
    p.requires_grad_(False)
visited = rnn.collect_hidden_states(model, n_trials=32, length=200, seed=7)
ic = fp.sample_initial_conditions(visited, n_ic=500, noise=0.1, seed=1)   # ICs from visited states
F = fp.discrete_velocity(model)
res = fp.find_fixed_points(F, ic, adam_steps=3000, lr=0.05, lbfgs_steps=30)
uniq = fp.dedup_cluster(res.points[res.speeds < fp.speed_tolerance(res.speeds)], tol=0.2)
Wout = model.w_out.weight.numpy(); bout = model.w_out.bias.numpy()
stable, saddle = [], []
for pt in uniq:
    (stable if fp.classify(fp.jacobian(F, pt)).label == 'stable' else saddle).append(pt)
print(f"unique fixed points: {len(uniq)}  ->  stable {len(stable)}, other {len(saddle)}")
print("GATE 8 stable at cube corners?",
      len(stable) == 8 and len({tuple(np.sign(Wout@p+bout).astype(int)) for p in stable}) == 8)

In [ ]:
# (3) Minimal viewer: readout-space trajectories + fixed points by stability class (validates the viz pipeline).
import plotly.graph_objects as go
with torch.no_grad():
    x_np, _ = rnn.generate_flipflop(6, 200, seed=42)
    _, out, _ = model(torch.from_numpy(x_np))
traj = out.numpy()
fig = go.Figure()
for k in range(traj.shape[0]):
    fig.add_trace(go.Scatter3d(x=traj[k,:,0], y=traj[k,:,1], z=traj[k,:,2],
                               mode='lines', line=dict(width=3), opacity=0.5, showlegend=False))
S = np.array([Wout@p+bout for p in stable])
Dd = np.array([Wout@p+bout for p in saddle]) if saddle else np.empty((0,3))
fig.add_trace(go.Scatter3d(x=S[:,0], y=S[:,1], z=S[:,2], mode='markers',
                           marker=dict(size=7, color='#2ecc71'), name='stable (8 corners)'))
if len(Dd):
    fig.add_trace(go.Scatter3d(x=Dd[:,0], y=Dd[:,1], z=Dd[:,2], mode='markers',
                               marker=dict(size=4, color='#e67e22'), name='saddle'))
fig.update_layout(template='plotly_dark', width=640, height=560,
                  title='Flip-flop: readout trajectories + fixed points',
                  scene=dict(xaxis_title='bit 1', yaxis_title='bit 2', zaxis_title='bit 3'))
fig.show()

**(4) Interpretation / 해석.** Gate passes: **8 stable fixed points at the 8 cube corners** (memory states) + **saddles on the edges/faces** between them — the Sussillo & Barak (2013) topology. Saddles are the transition structure (a pulse pushes the state over a saddle into a neighbouring corner-basin), so readout trajectories run along cube edges. Differential test: the reference finder on the same net lands on our corners to **5e-4** (precision), coverage 6/8 (reference GD is slow; our finder gets all 8). **Failure modes watched:** ICs from visited states (not uniform); speed cutoff from the q-distribution; clustering dedup.

**한국어.** 게이트 통과: 큐브 꼭짓점 8개의 안정 고정점 + 사이의 안장점(전이 구조) — Sussillo & Barak 위상. 차등 검증: 레퍼런스가 우리 꼭짓점에 5e-4로 일치, 커버리지 6/8(레퍼런스 GD가 느림; 우리 탐색기는 8개 모두).

## M2 · Brain data — PCA + jPCA (Stage 1) _(PASS — tests/test_m2_brain.py)_
**(1) math/why:** MC_Maze is blocked here (DANDI API 403 by egress policy), so we use the **actual Stage-1 dataset** — Churchland et al. 2012, 218 neurons × 108 reach conditions (public, and read by the reference oracle's loader). Rates are already trial-averaged/smoothed and aligned to movement onset; we apply the M0 jPCA pipeline over the classic −50..150 ms window.
**Adversarial:** does the rotation survive *without* cross-condition-mean subtraction? Brain viewer: trajectories + jPCA plane, **NO fixed points** (no equations for the brain).

**한국어.** MC_Maze는 이 환경에서 차단(DANDI API 403) → Stage-1 원 데이터인 Churchland 2012(뉴런 218 × 조건 108) 사용. M0 파이프라인을 −50..150 ms 창에 적용. 적대적 검증: 조건간 평균 차감 없이도 회전이 남는가? 뇌 뷰어: 궤적 + jPCA 평면, **고정점 없음**.

In [ ]:
# 🔍 구현 핵심 — jPCA 전처리(step 2): 소프트 정규화 + 조건간 평균 차감(Lebedev 2019 논쟁 단계).
# jPCA preprocessing (step 2): soft-normalization + the cross-condition-mean subtraction
# that Lebedev et al. (2019) contested. M2 tests adversarially whether rotation survives without it.
import inspect
from python import pca_jpca as J
print(inspect.getsource(J.soft_normalize))
print(inspect.getsource(J.subtract_cc_mean))

In [ ]:
# (2) Load real M1 data (downloads to cache on first use), run jPCA + differential test.
import numpy as np
from python import data as D, pca_jpca as J
rd = D.load_churchland(cache_dir=str(CACHE_DIR))
m = (rd.times >= -50) & (rd.times <= 150); X = rd.rates[:, m, :]; tw = rd.times[m]
dt_s = (tw[1] - tw[0]) / 1000.0
res = J.jpca(X, num_pcs=6, subtract_ccm=True)
print(f"data {rd.rates.shape} (C,T,N)  |  {rd.source}")
print(f"fit R2: {res.fit_R2:.4f}   top-plane var: {res.plane_var_frac:.4f}   freq: {res.omega/(2*np.pi*dt_s):.2f} Hz")
# differential test vs Antin on identical real data:  %run tests/test_m2_brain.py

In [ ]:
# (3) Inline plotly: 108 conditions on the jPCA rotation plane (the Churchland 2012 result).
import plotly.graph_objects as go
fig = go.Figure()
C = res.projected.shape[0]
for c in range(C):
    xy = res.projected[c]
    fig.add_trace(go.Scatter(x=xy[:,0], y=xy[:,1], mode='lines',
                             line=dict(width=1.3, color=f'hsl({int(360*c/C)},70%,55%)'),
                             opacity=0.75, showlegend=False))
    fig.add_trace(go.Scatter(x=[xy[0,0]], y=[xy[0,1]], mode='markers',
                             marker=dict(size=4, color=f'hsl({int(360*c/C)},70%,55%)'), showlegend=False))
fig.update_layout(template='plotly_dark', width=600, height=560,
                  title=f'Brain jPCA plane — Churchland 2012 (R2={res.fit_R2:.2f}, planeVar={res.plane_var_frac:.2f})',
                  xaxis_title='jPC1', yaxis_title='jPC2')
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

In [ ]:
# (3b) Adversarial: rotation WITHOUT cross-condition-mean subtraction (Lebedev 2019 critique).
res0 = J.jpca(X, num_pcs=6, subtract_ccm=False)
print(f"fit R2       no-CCM {res0.fit_R2:.4f}  vs  CCM {res.fit_R2:.4f}")
print(f"plane var    no-CCM {res0.plane_var_frac:.4f}  vs  CCM {res.plane_var_frac:.4f}")
print(f"frequency    no-CCM {res0.omega/(2*np.pi*dt_s):.2f} Hz  vs  CCM {res.omega/(2*np.pi*dt_s):.2f} Hz")
# export the brain JSON for the web viewer (NO fixed points)
from python import export as E
E.export_brain(str(DATA_DIR/'brain.json'), res, rd.conditions,
               meta={'source': rd.source, 'window_ms': [-50,150],
                     'freq_hz': round(float(res.omega/(2*np.pi*dt_s)),3)})
print('wrote', DATA_DIR/'brain.json')

**(4) Interpretation / 해석.** Faithful reproduction: with CCM subtraction the 108 conditions start clustered and sweep into a coherent rotational fan — **fit R² 0.52, top plane 46% of variance, ~1.4 Hz**. The hand pipeline matches Antin on this real data to **0.0001°** (differential test on brain input). **Adversarial result:** the rotation *survives* without CCM subtraction (R² 0.56, plane 56%) — so it is **not manufactured** by that step (contra a strong reading of Lebedev 2019); but the frequency shifts to ~2.5 Hz because, without CCM, jPCA characterizes the large condition-*independent* movement signal instead of the condition-*dependent* rotation that is Churchland's actual claim. The brain export carries **no fixed points** — we have no equations for the brain.

**한국어.** 충실한 재현: CCM 차감 시 108 조건이 회전 부채꼴을 그린다(R² 0.52, 평면 46%, ~1.4 Hz). 실데이터에서 Antin과 0.0001° 일치. 적대적 결과: CCM 없이도 회전이 남는다(R² 0.56) → CCM이 회전을 *만들어내지 않음*. 다만 주파수는 ~2.5 Hz로 바뀌는데, CCM 없이는 조건-*독립* 신호를 특성화하기 때문. 뇌 내보내기는 **고정점 없음**.

## M3 · Reaching RNN + jPCA (Stage 2 part 1) _(PASS — tests/test_m3_reaching.py)_
**(1) math/why:** 256-unit continuous-time tanh RNN, `dx/dt = -x + W_rec·φ(x) + W_in·u + b`, `z = W_out·x`; inputs = target + go, output = hand velocity; **metabolic L2-on-rates kept (README-mandatory)**. Trained on the **TASK, not on spikes** — then we ask whether the *emergent* hidden dynamics rotate like M1. (Conflating this with "fit the RNN to spikes" is the classic fatal error.)
**Task gate:** velocity R² above threshold BEFORE any dynamics analysis. Then jPCA on condition-averaged hidden states over the movement epoch.

**한국어.** 256유닛 연속시간 tanh RNN, 과제(목표+go → 손 속도)로 학습(**스파이크가 아님**), 발화율 L2 정규화 유지(README 필수). 과제 게이트: 속도 R² 임계값 이상일 때만 동역학 분석. 그다음 움직임 구간 은닉 상태에 jPCA — RNN도 회전하는가?

In [ ]:
# 🔍 구현 핵심 — 연속시간 도달 RNN (dx/dt = -x + W_rec φ(x) + W_in u + b).
# The continuous-time reaching RNN. `velocity(x, u)` = dx/dt is the closed-form field
# the M4 fixed-point finder later operates on — this is why the RNN gets a mechanistic layer.
import inspect
from python import rnn
print(inspect.getsource(rnn.ReachingRNN))

In [ ]:
# (2) Train the reaching RNN on the TASK (not spikes), gate on velocity R2, then jPCA the hidden states.
import numpy as np, torch
from python import rnn, pca_jpca as J

model, vel_r2, _ = rnn.train_reaching(n_hid=256, iters=800, n_dirs=16, reps=8,
                                      lr=2e-3, metabolic=1e-3, seed=0)      # ~90s in Colab
print(f"velocity R2: {vel_r2:.4f}  (task gate: > 0.90)")
assert vel_r2 > 0.90, "task not solved — analyzing failed-task dynamics is meaningless"

Xc, _ = rnn.reaching_condition_averaged(model, n_dirs=16, T=60, go=20)      # (16, T, 256) hidden states
res = J.jpca(Xc[:, 15:45, :], num_pcs=6, subtract_ccm=True)                 # movement epoch
print(f"jPCA on RNN hidden states -> fit R2 {res.fit_R2:.3f}, top-plane var {res.plane_var_frac:.3f}")
print("=> emergent hidden dynamics ROTATE" if res.fit_R2 > 0.5 else "=> no rotation")

In [ ]:
# (3) Inline plotly: RNN hidden-state trajectories on their own jPC plane — do they rotate?
import plotly.graph_objects as go
fig = go.Figure()
C = res.projected.shape[0]
for c in range(C):
    xy = res.projected[c]
    fig.add_trace(go.Scatter(x=xy[:, 0], y=xy[:, 1], mode='lines',
                             line=dict(width=1.6, color=f'hsl({int(360*c/C)},70%,55%)'),
                             opacity=0.85, showlegend=False))
    fig.add_trace(go.Scatter(x=[xy[0, 0]], y=[xy[0, 1]], mode='markers',
                             marker=dict(size=5, color=f'hsl({int(360*c/C)},70%,55%)'), showlegend=False))
fig.update_layout(template='plotly_dark', width=600, height=560,
                  title=f'Reaching RNN jPCA plane (fitR2={res.fit_R2:.2f}) — emergent rotation',
                  xaxis_title='jPC1', yaxis_title='jPC2')
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

**(4) Interpretation / 해석.** The RNN solves the task (**velocity R² 0.999**) and its *emergent* hidden dynamics **rotate** — skew fit R² **0.88**, the same signature jPCA found in the brain, even though the network was never shown a spike. This is the Sussillo et al. 2015 result reproduced: rotational population dynamics arise from the *demands of the reach task*, not from copying M1. Note the top rotation plane holds only ~15% of hidden variance: the largest variance direction is the (static) target-tuning axis; the rotation is a real but lower-variance component (fit R² is high because the *dynamics* — the derivative structure — are strongly rotational).

**Honesty note (metabolic reg).** The README mandates the metabolic penalty because *without* it the network supposedly finds a non-biological high-dimensional solution. In this simplified 16-direction task that effect **does not reproduce**: an ablation leaves the participation ratio essentially unchanged (~2.5 with and without). The task is simple enough that even the unregularized solution is low-dimensional. We keep the penalty for faithfulness and report the null result honestly — see NOTES "M3" / "Proposed deviations".

**한국어.** RNN은 과제를 풀고(속도 R² 0.999) *창발적* 은닉 동역학이 **회전**한다(적합 R² 0.88) — 스파이크를 준 적 없는데도 뇌와 같은 신호. Sussillo 2015 재현: 회전은 M1을 베낀 게 아니라 *도달 과제의 요구*에서 나온다. 최상위 회전 평면은 은닉 분산의 ~15%뿐 — 최대 분산 축은 (정적) 목표 튜닝 축이고 회전은 실재하지만 분산이 작은 성분(동역학=미분 구조가 강하게 회전적이라 적합 R²는 높다).

**정직성 노트(대사 정규화).** README는 대사 페널티를 필수로 두는데, 이유는 *없으면* 비생물학적 고차원 해가 나온다는 것. 이 단순화된 16방향 과제에선 그 효과가 **재현되지 않는다**: 절제 실험에서 참여율(PR)이 사실상 그대로(~2.5)다. 과제가 단순해 무정규화 해도 저차원이기 때문. 충실성을 위해 페널티는 유지하고 이 무효 결과를 정직하게 보고한다 — NOTES 참조.

## M4 · Reaching RNN fixed points + flow field (Stage 2 part 2) _(PASS — tests/test_m4_fixedpoints.py)_
**(1) math/why:** minimize `q(x)=½‖dx/dt‖²` on the **autonomous** field `F(x) = −x + W_rec φ(x)` (Adam + L-BFGS); ICs **only from movement-epoch states + noise**; tolerance from the q-**distribution**; de-dup by clustering; classify by Jacobian eigenvalues; sample `dx/dt` on a grid in the jPCA plane.
**Adversarial:** fixed points stable to IC re-sampling? does an independent solver agree? (The M1 discrete reference can't analyze a continuous RNN → SciPy Newton is the oracle.) RNN viewer: trajectories + flow + fixed point by class.

**한국어.** 자율 벡터장 `F(x)=−x+W_rec φ(x)`에 `q=½‖dx/dt‖²` 최소화(Adam+L-BFGS); IC는 움직임 구간 상태+노이즈만; 허용오차는 q 분포; 군집화 중복 제거; 야코비안 고유값으로 분류; jPCA 평면 격자에 흐름장. 적대적: IC 재샘플링 안정성 + 독립 솔버(SciPy Newton) 일치. 뷰어: 궤적 + 흐름장 + 분류별 고정점.

In [ ]:
# 🔍 구현 핵심 — 야코비안 안정성 분류 + jPCA 평면 위 흐름장 샘플링.
# Jacobian stability classification (stable/unstable/saddle/rotational; the leading complex
# eigenpair is the local rotation rate) and sampling dx/dt on a grid in the jPCA plane.
import inspect
from python import fixedpoints as fp, flowfield as ff
print(inspect.getsource(fp.classify))
print(inspect.getsource(ff.flow_on_grid))

In [ ]:
# (2) Fixed points of the reaching RNN's AUTONOMOUS field F(x) = -x + W_rec tanh(x). Reuses the M3 model.
import numpy as np, torch
from python import rnn, fixedpoints as fp, flowfield as ff, pca_jpca as J
assert isinstance(model, rnn.ReachingRNN), "run the M3 cell first (defines the reaching `model`)"
for p in model.parameters():
    p.requires_grad_(False)
uc = torch.zeros(model.n_in)
F = lambda x: model.velocity(x, uc)                    # autonomous velocity (input held at 0)

with torch.no_grad():                                  # ICs from movement-epoch states (NOT uniform)
    u_np, _, _ = rnn.generate_reaching(16, reps=6, T=60, go=20, seed=3)
    Xv, _ = model(torch.from_numpy(u_np))
vis = Xv[:, 22:45, :].reshape(-1, 256).numpy()
ic = fp.sample_initial_conditions(vis, n_ic=800, noise=0.3, seed=1)
fpr = fp.find_fixed_points(F, ic, adam_steps=4000, lr=0.02, lbfgs_steps=40)
uniq = fp.dedup_cluster(fpr.points[fpr.speeds < fp.speed_tolerance(fpr.speeds)], tol=1.0)
print(f"unique fixed points: {len(uniq)}")
fps = []
for pt in uniq:
    st = fp.classify(fp.jacobian(F, pt), real_tol=1e-2)
    fps.append((pt, st))
    print(f"  |x|={np.linalg.norm(pt):.2f}  {st.label}  spiral={st.is_spiral}  "
          f"lead eig={st.lead_eig.real:+.2f}{st.lead_eig.imag:+.2f}j  (Im = local rotation rate)")

In [ ]:
# (3) RNN viewer: flow field (dx/dt) + trajectories + fixed point, all in the jPCA plane.
import plotly.figure_factory as ffx
import plotly.graph_objects as go

with torch.no_grad():
    u1, _, _ = rnn.generate_reaching(16, reps=1, T=60, go=20, noise=0.0, seed=0)
    Xc, _ = model(u1 if torch.is_tensor(u1) else torch.from_numpy(u1))
jr = J.jpca(Xc.numpy()[:, 15:45, :], num_pcs=6, subtract_ccm=True)
origin, plane = ff.jpca_plane_in_hidden(jr)             # hidden-space plane => one shared projection
field = ff.flow_on_grid(F, origin, plane, extent=3.5, n=17)

qx, qy = field.coords[:, 0], field.coords[:, 1]
qu, qv = field.vectors[:, 0], field.vectors[:, 1]
fig = ffx.create_quiver(qx, qy, qu, qv, scale=0.03, line=dict(color='#5b8def', width=1), name='flow dx/dt')
Xh = Xc.numpy()[:, 15:45, :]
for c in range(16):
    xy = (Xh[c] - origin) @ plane
    fig.add_trace(go.Scatter(x=xy[:, 0], y=xy[:, 1], mode='lines',
                             line=dict(width=1.6, color=f'hsl({int(360*c/16)},75%,55%)'), showlegend=False))
S = np.array([(pt - origin) @ plane for pt, _ in fps])
fig.add_trace(go.Scatter(x=S[:, 0], y=S[:, 1], mode='markers', name='rotational saddle',
                         marker=dict(symbol='x', size=14, color='crimson', line=dict(width=2))))
fig.update_layout(template='plotly_dark', width=680, height=620,
                  title='Reaching RNN — flow field + fixed point (the rotation generator)',
                  xaxis_title='jPC1', yaxis_title='jPC2')
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

**(4) Interpretation / 해석.** This is the payoff. The autonomous field has **one dominant fixed point — a rotational saddle** (leading eigenpair **+0.51 ± 1.33i**), and the flow spirals around it while the condition trajectories spiral *outward* along that flow. The imaginary part (~1.33) is the local rotation rate: **the complex eigenpair GENERATES the rotation that jPCA merely DESCRIBED at M2/M3.** Descriptive (jPCA) and mechanistic (fixed point) are now joined — one explains the other. **Contrast M1:** the flip-flop had **8 point attractors** (discrete memory); the reach RNN has **1 rotational saddle** (a timed rotation generator). Same finder, opposite topology, because the computations differ. **Adversarial:** the point survives IC re-sampling (re-found within 0.038) and an independent SciPy Newton solve (residual 2e-7, distance 0.037) — not an artifact. Because this is the RNN, we *can* do all this; the brain (no equations) gets trajectories + plane only.

**한국어.** 핵심 결론. 자율 벡터장에 **지배적 고정점 하나 — 회전 안장점**(주고유쌍 **+0.51 ± 1.33i**)이 있고, 흐름은 그 주위를 나선형으로 돌며 조건 궤적은 그 흐름을 따라 바깥으로 나선을 그린다. 허수부(~1.33)가 국소 회전율이다: **복소 고유쌍이 M2/M3에서 jPCA가 *기술*하던 회전을 *생성*한다.** 기술적(jPCA)과 기전적(고정점)이 이제 연결된다. **M1 대비:** 플립플롭은 점 끌개 8개(이산 기억), 도달 RNN은 회전 안장점 1개(시간 회전 생성기) — 같은 탐색기, 반대 위상, 계산이 다르기 때문. **적대적:** IC 재샘플링(0.038)·독립 Newton(잔차 2e-7, 거리 0.037)로 확인 — 인공물 아님. RNN이라 이 모든 게 가능; 뇌(방정식 없음)는 궤적+평면만.

## M5 · Unified viewer + deploy _(DONE — web/index.html, verified)_
**(1) math/why:** toggle Brain | RNN | side-by-side. The two live in **different spaces** — each rendered in its own jPCA space, matched by visual scale only; the viewer says so and disables the mechanism layers (flow, fixed points) for the brain, because we have no equations for it. Export JSON via `export.py`; the viewer is a single self-contained `web/index.html` (data inlined, no external libs) → works as a local file, on **Hugging Face Spaces (Static SDK)**, and as a shareable artifact.

**한국어.** 뇌 | RNN | 나란히 토글. 둘은 서로 다른 공간 — 각자 jPCA 공간에서 렌더링, 시각 스케일만 맞춤; 뷰어가 이를 명시하고 뇌에는 기전 레이어(흐름/고정점)를 비활성화한다(방정식이 없으므로). `export.py`로 JSON 내보내고, 뷰어는 데이터가 인라인된 단일 `web/index.html`(외부 라이브러리 없음) — 로컬 파일·HF Spaces·공유 아티팩트로 동작.

In [ ]:
# (2) Export both datasets and build the self-contained viewer.
# Brain: trajectories + jPCA plane only (no mechanism). RNN: + fixed points + flow.
import subprocess, sys
from python import data as D, pca_jpca as J, export as E

# brain.json (Stage 1) — no fixed points / no flow (we have no equations for the brain)
rd = D.load_churchland(cache_dir=str(CACHE_DIR))
mask = (rd.times >= -50) & (rd.times <= 150)
res_brain = J.jpca(rd.rates[:, mask, :], num_pcs=6, subtract_ccm=True)
E.export_brain(str(DATA_DIR / 'brain.json'), res_brain, rd.conditions,
               meta={'source': rd.source, 'window_ms': [-50, 150]})

# rnn.json (Stage 2) — trajectories + fixed point(s) + flow field, one shared basis
subprocess.run([sys.executable, str(REPO / 'scripts' / 'build_rnn_export.py')], check=True)
# inline both JSONs into a single self-contained web/index.html
subprocess.run([sys.executable, str(REPO / 'scripts' / 'build_viewer.py')], check=True)
print('Built web/index.html — open it, or deploy to Hugging Face Spaces (see web/README.md).')

## 정리 / What you have learned

You walked the full arc, in order:

- **M0** — jPCA by hand: fit one global **skew-symmetric** system to trajectory
  geometry (matches the reference oracle to <0.001°). *Descriptive.*
- **M1** — a **known-answer** instrument: the 3-bit flip-flop has exactly **8**
  stable fixed points at cube corners. This calibrates the fixed-point finder
  before trusting it on anything unknown. *Mechanistic.*
- **M2** — the same jPCA on **real brain data** (Churchland 2012): rotation is
  real and *survives* removing the contested preprocessing step, but that step
  changes *which* rotation you measure.
- **M3** — a reaching RNN trained **on the task, not on spikes**, rotates too —
  rotation is an emergent consequence of the computation.
- **M4** — the payoff: the RNN's rotation is **generated** by a single
  **rotational saddle** whose complex eigenpair (0.51±1.33i) *is* the local
  rotation. Descriptive (jPCA) and mechanistic (fixed points) are now joined.
- **M5** — a viewer that shows the brain/RNN **asymmetry** honestly: the brain
  gets trajectories + a plane; only the RNN gets fixed points + flow.

**To go deeper:** each milestone has a falsifiable test in `tests/` (run them to
see the numbers reproduce), and `NOTES.md` records the *why* and the honest
findings (metabolic-reg mismatch, MC_Maze→Churchland). The scope deliberately
stops here; `NOTES.md → "Next iteration"` lists what comes next (null models, DSA).

---

**한국어.** 전체 흐름을 순서대로 통과했습니다: **M0** jPCA 직접 구현(기술적) → **M1**
정답 아는 플립플롭으로 탐색기 보정(8 꼭짓점) → **M2** 실제 뇌 데이터 재현(전처리 논쟁까지)
→ **M3** 과제로 학습한 RNN도 회전 → **M4** 그 회전을 *생성*하는 회전 안장점(0.51±1.33i)
→ **M5** 뇌/RNN 비대칭을 정직하게 보여주는 뷰어. 더 깊이: `tests/`로 숫자 재현 확인,
`NOTES.md`로 이유·정직한 발견 확인. 다음 단계는 `NOTES.md "Next iteration"` 참조.